# Text-to-SQL QLoRA Fine-Tuning
Fine-tunes Llama-3.2-3B-Instruct on Spider benchmark using QLoRA.
**GPU:** Enable via Settings → Accelerator → GPU T4 x2

In [ ]:
# ── Install dependencies ──────────────────────────────────────
!pip install -q --upgrade --no-deps bitsandbytes
!pip install -q --upgrade transformers accelerate peft

import importlib, accelerate, peft, bitsandbytes as bnb, torch
print(f"bitsandbytes : {bnb.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"peft         : {peft.__version__}")
print(f"PyTorch      : {torch.__version__}  |  CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# ── HuggingFace login (optional — Qwen2.5 is ungated) ────────
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace.")
except Exception:
    print("No HF token found — continuing without login (ok for Qwen2.5).")

In [ ]:
# ── Check GPU ────────────────────────────────────────────────
import torch
print(f'GPUs available: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name}  {props.total_memory / 1e9:.1f} GB')

In [ ]:
# ── Prepare dataset ──────────────────────────────────────────
# Uses b-mc2/sql-create-context: 78k examples, every row has real CREATE TABLE schema
import json, os
from datasets import load_dataset

INSTRUCTION = (
    'Given the database schema and a natural language question, '
    'write the correct SQL query.\n\n'
    'Schema:\n{schema}\n\n'
    'Question: {question}\n\nSQL:'
)

print('Loading sql-create-context...')
ds = load_dataset('b-mc2/sql-create-context', split='train')
print(f'Total: {len(ds)} examples')

split    = ds.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
val_ds   = split['test']

def make_examples(dataset):
    return [{
        'prompt':     INSTRUCTION.format(schema=row['context'], question=row['question']),
        'completion': row['answer'].strip(),
    } for row in dataset]

train_data = make_examples(train_ds)
val_data   = make_examples(val_ds)

os.makedirs('/kaggle/working/data', exist_ok=True)
with open('/kaggle/working/data/train.jsonl', 'w') as f:
    for ex in train_data: f.write(json.dumps(ex) + '\n')
with open('/kaggle/working/data/val.jsonl', 'w') as f:
    for ex in val_data: f.write(json.dumps(ex) + '\n')

# Verify real schema is being used
print(f'\nSample prompt:\n{train_data[0]["prompt"][:350]}\n...')
print(f'\nTrain: {len(train_data)} | Val: {len(val_data)}')

In [ ]:
# ── Config ──────────────────────────────────────────────────
BASE_MODEL   = 'Qwen/Qwen2.5-3B-Instruct'   # fully open, no gating
OUTPUT_DIR   = '/kaggle/working/checkpoints'
TRAIN_PATH   = '/kaggle/working/data/train.jsonl'
VAL_PATH     = '/kaggle/working/data/val.jsonl'
MAX_LENGTH   = 512
BATCH_SIZE   = 4
GRAD_ACCUM   = 4
EPOCHS       = 3
LR           = 2e-4
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
SEED         = 42

LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                 'gate_proj', 'up_proj', 'down_proj']

# Qwen2.5 uses ChatML format
TEMPLATE = (
    '<|im_start|>system\n'
    'You are a Text-to-SQL assistant. Convert natural language questions '
    'to correct SQL queries based on the given schema.<|im_end|>\n'
    '<|im_start|>user\n'
    '{prompt}<|im_end|>\n'
    '<|im_start|>assistant\n'
    '{completion}<|im_end|>'
)

In [ ]:
# ── Load model + tokenizer ───────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    dtype=torch.float16,       # torch_dtype is deprecated, use dtype
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Model loaded.')

In [ ]:
# ── Apply LoRA ───────────────────────────────────────────────
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Tokenize dataset ─────────────────────────────────────────
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

def tokenize(examples):
    ids_list, masks, labels_list = [], [], []
    for prompt, completion in zip(examples['prompt'], examples['completion']):
        full = TEMPLATE.format(prompt=prompt, completion=completion)
        tok  = tokenizer(full, max_length=MAX_LENGTH, truncation=True, padding=False)
        ids  = tok['input_ids']

        prompt_only = TEMPLATE.format(prompt=prompt, completion='')
        p_len = len(tokenizer(prompt_only, max_length=MAX_LENGTH, truncation=True)['input_ids'])

        ids_list.append(ids)
        masks.append(tok['attention_mask'])
        labels_list.append([-100] * p_len + ids[p_len:])

    return {'input_ids': ids_list, 'attention_mask': masks, 'labels': labels_list}

train_ds = Dataset.from_list(load_jsonl(TRAIN_PATH))
val_ds   = Dataset.from_list(load_jsonl(VAL_PATH))

train_ds = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(tokenize,   batched=True, remove_columns=val_ds.column_names)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# ── Train ────────────────────────────────────────────────────
import math, os
from transformers import (
    TrainingArguments, Trainer,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
)

use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16
print(f"bf16: {use_bf16}  |  fp16: {use_fp16}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    bf16=use_bf16,
    fp16=use_fp16,
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    seed=SEED,
    dataloader_num_workers=2,
    optim='paged_adamw_8bit',
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, pad_to_multiple_of=8)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('Starting training...')
trainer.train()

final_path = os.path.join(OUTPUT_DIR, 'final')
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print(f'Adapter saved → {final_path}')

metrics    = trainer.evaluate()
perplexity = math.exp(metrics['eval_loss'])
print(f'Eval loss  : {metrics["eval_loss"]:.4f}')
print(f'Perplexity : {perplexity:.2f}')

In [ ]:
# ── Save adapter as a Kaggle dataset output ──────────────────
# Everything in /kaggle/working/ is automatically saved as output
import shutil
shutil.make_archive('/kaggle/working/lora_adapter', 'zip', final_path)
print('lora_adapter.zip ready for download in Kaggle output panel.')